In [13]:
import pandas as pd
import numpy as np
from functools import reduce
import os

In [14]:
# 加载原始数据
pop_df = pd.read_csv('data/data_raw/population.csv')

# 识别所有包含人口数值的列 (例如 population_2011, population_2012 等)
pop_cols = [c for c in pop_df.columns if 'population_' in c]

# 清洗千分位逗号并强制转换为整数 (int)
for col in pop_cols:
    # .str.replace(',', '')：去掉数字中的逗号，防止 Python 将其视为字符串
    # .astype(int)：将清洗后的文本强制转换为整数类型，确保计算精度且不带小数点
    pop_df[col] = pop_df[col].astype(str).str.replace(',', '').astype(int)

# 执行逆透视转换：将“宽表”转为“长表”
pop_long = pd.melt(
    pop_df,
    id_vars=['ladcode23'],     # 保持不变的标识符列
    value_vars=pop_cols,       # 需要转置的年份列
    var_name='year',           # 新列：存放 'population_2011' 等名称
    value_name='pop_value'     # 新列：存放人口数值
)

# 4. 清洗年份和代码
# 去掉 'population_' 前缀只保留数字，并去除地区代码可能存在的空格
pop_long['year'] = pop_long['year'].str.replace('population_', '').astype(int)
pop_long['ons_code'] = pop_long['ladcode23'].str.strip()

# 5. 聚合计算：得到每个地区每年的总人口 (LAD级别)
# 由于原始数据是按性别和年龄分行的，通过求和将它们合并
pop_annual = pop_long.groupby(['ons_code', 'year'])['pop_value'].sum().reset_index()

# 6. 确保最终结果也是整数类型
pop_annual['pop_value'] = pop_annual['pop_value'].astype(int)

# 检查处理后的数据
print("处理后的人口数据 (前5行):")
print(pop_annual.head())
pop_annual.sort_values(['ons_code', 'year']).to_csv('data/data_cleaned/population_cleaned.csv', index=False)


处理后的人口数据 (前5行):
    ons_code  year  pop_value
0  E06000001  2011      92088
1  E06000001  2012      92344
2  E06000001  2013      92465
3  E06000001  2014      92358
4  E06000001  2015      92125


In [15]:
# 1. 加载充电桩数据作为分级参考
charging = pd.read_csv('data/data_raw/Charging_Port.csv').dropna(how='all')
charging.columns = [c.strip() for c in charging.columns]

# 2. 初始化状态机变量
hierarchy = []
curr_ctry = None  # 当前构成国
curr_reg = None   # 当前大区
curr_cty = None   # 当前郡

# 3. 开始逐行扫描
for _, row in charging.iterrows():
    # 排除空行或无效编码
    if pd.isna(row['ONS_code']) or str(row['ONS_code']).strip().lower() == 'nan':
        continue

    code = str(row['ONS_code']).strip()
    name = str(row['ONS_geo']).strip()
    lvl = str(row['Geo_level']).strip()

    # 状态机：更新构成国、大区、郡的记忆
    if lvl == 'Country':
        curr_ctry = name
        # 统一让 Country 的 Region 记忆等于它自己
        curr_reg = name
        if name != 'England':
            # 威尔士、苏格兰、北爱没有 Region，升格自己为 County 占位
            curr_cty = name
        else:
            # 英格兰下面有具体郡，先清空记忆等待后续更新
            curr_cty = None

    elif lvl == 'Region':
        curr_reg = name
        curr_cty = name

    elif lvl == 'County':
        curr_cty = name

    # 定义 UK 和 GB 的包含逻辑
    # 只要是 E, W, S, N 开头的区域，都属于 United Kingdom
    p_uk = "United Kingdom"

    # 只有 E, W, S 开头的区域属于 Great Britain
    # N 开头的区域 p_gb 设为 None
    if code.startswith(('E', 'W', 'S', 'K03')):
        p_gb = "Great Britain"
    else:
        p_gb = None

    # --- 存储结果 ---
    valid_lvls = ['National', 'Country', 'Region', 'County', 'LAD']
    if lvl in valid_lvls:
        # 特殊处理：如果本身就是 National 行，手动纠正归属
        if lvl == 'National':
            # United Kingdom 行不属于 GB
            if name == 'United Kingdom': p_gb = None
            # Great Britain 行本身属于 UK
            if name == 'Great Britain': p_uk = 'United Kingdom'; p_gb = None

        hierarchy.append({
            'ONS_code': code,
            'ONS_geo': name,
            'Geo_level': lvl,
            'Country': curr_ctry,
            'Region': curr_reg,
            'p_uk': p_uk,     # 联合王国标签
            'p_gb': p_gb,     # 大不列颠标签
            'p_cty': curr_cty
        })

# 4. 转换为 DataFrame 形式的分级字典
geo_lookup = pd.DataFrame(hierarchy)

# 验证逻辑：检查北爱尔兰（N开头）是否不在 GB 下面
print("北爱尔兰样本检查：")
print(geo_lookup[geo_lookup['ONS_code'].str.startswith('N', na=False)].head(3))

# 验证逻辑：检查苏格兰（S开头）是否同时在 UK 和 GB 下面
print("\n苏格兰样本检查：")
print(geo_lookup[geo_lookup['ONS_code'].str.startswith('S', na=False)].head(3))

# 导出
geo_lookup.to_csv('data/data_cleaned/geo_lookup.csv', index=False)

北爱尔兰样本检查：
      ONS_code                  ONS_geo Geo_level           Country  \
421  N92000002         Northern Ireland   Country  Northern Ireland   
422  N09000001  Antrim and Newtownabbey       LAD  Northern Ireland   
423  N09000011      Ards and North Down       LAD  Northern Ireland   

               Region            p_uk p_gb             p_cty  
421  Northern Ireland  United Kingdom  NaN  Northern Ireland  
422  Northern Ireland  United Kingdom  NaN  Northern Ireland  
423  Northern Ireland  United Kingdom  NaN  Northern Ireland  

苏格兰样本检查：
      ONS_code        ONS_geo Geo_level   Country    Region            p_uk  \
388  S92000003       Scotland   Country  Scotland  Scotland  United Kingdom   
389  S12000033  Aberdeen City       LAD  Scotland  Scotland  United Kingdom   
390  S12000034  Aberdeenshire       LAD  Scotland  Scotland  United Kingdom   

              p_gb     p_cty  
388  Great Britain  Scotland  
389  Great Britain  Scotland  
390  Great Britain  Scotland  


In [16]:
# 1. 设定路径与指标映射
excel_path = 'data/data_raw/regional_gross_disposable_household_income_local_authorities_2023.xlsx'
sheet_metrics = {'Table 3': 'gdhi_per_head', 'Table 4': 'gdhi_index', 'Table 6': 'gdhi_growth'}
years = list(range(2011, 2024))

# 2. 一次性读取三个 Sheet
all_sheets = pd.read_excel(excel_path, sheet_name=list(sheet_metrics.keys()), header=1)

# 3. 批量清洗与逆透视 (Melt)
processed_list = []
for sheet_name, metric in sheet_metrics.items():
    df = all_sheets[sheet_name]
    # 筛选 LAD 级别行
    df = df[df['LAD code'].astype(str).str.len() == 9].copy()
    df.columns = [str(col).strip() for col in df.columns]
    years_str = [str(y) for y in years]

    df_melted = df.melt(id_vars=['LAD code'], value_vars=years_str, var_name='year', value_name=metric)
    df_melted['year'] = df_melted['year'].astype(int)
    df_melted['ons_code'] = df_melted['LAD code'].str.strip()
    processed_list.append(df_melted[['ons_code', 'year', metric]])

# 4. 合并指标并挂载人口权重
gdhi_lad_base = reduce(lambda left, right: pd.merge(left, right, on=['ons_code', 'year']), processed_list)
# 使用 how='left' 确保没有人口数据的 S/N 地区不被过滤
gdhi_weighted = pd.merge(gdhi_lad_base, pop_annual, on=['ons_code', 'year'], how='left')

# 计算加权分子 (总量 = 人均 * 人口)
gdhi_weighted['w_head'] = gdhi_weighted['gdhi_per_head'] * gdhi_weighted['pop_value']
gdhi_weighted['w_index'] = gdhi_weighted['gdhi_index'] * gdhi_weighted['pop_value']

# 5. 关联地理字典
lad_full = pd.merge(gdhi_weighted, geo_lookup[geo_lookup['Geo_level'] == 'LAD'],
                    left_on='ons_code', right_on='ONS_code', how='inner')

# 6. 执行多层级加权聚合
agg_results = []

# 定义聚合任务列表：(目标层级名称, 字典中的分组字段)
agg_tasks = [
    ('National', 'p_uk'),  # 聚合全英国数据
    ('National', 'p_gb'),  # 聚合大不列颠数据
    ('Country', 'Country'),
    ('Region', 'Region'),
    ('County', 'p_cty')
]

for lvl, group_key in agg_tasks:
    # 按照指定的 group_key 分组
    agg = lad_full.groupby([group_key, 'year']).agg({
        'w_head': 'sum',
        'w_index': 'sum',
        'pop_value': 'sum'
    }).reset_index()

    # 计算加权平均值，处理分母为 0 的情况
    agg['gdhi_per_head'] = np.where(agg['pop_value'] > 0, agg['w_head'] / agg['pop_value'], np.nan)
    agg['gdhi_index'] = np.where(agg['pop_value'] > 0, agg['w_index'] / agg['pop_value'], np.nan)

    # 匹配 ONS 信息。
    meta = geo_lookup[geo_lookup['Geo_level'] == lvl][['ONS_code', 'ONS_geo', 'Country', 'Region', 'Geo_level']].drop_duplicates()
    agg = pd.merge(agg.rename(columns={group_key: 'ONS_geo'}), meta, on='ONS_geo', how='inner')

    # 重新计算增长率
    agg = agg.sort_values(['ONS_code', 'year'])
    agg['gdhi_growth'] = agg.groupby('ONS_code')['gdhi_per_head'].pct_change(fill_method=None) * 100
    agg_results.append(agg)

# 7. 最终合并与格式化
final_cols = ['ONS_code', 'ONS_geo', 'Country', 'Region', 'Geo_level', 'year', 'gdhi_per_head', 'gdhi_index', 'gdhi_growth']
gdhi_master = pd.concat([lad_full[final_cols]] + [df[final_cols] for df in agg_results], ignore_index=True)

# 数值精度整理
gdhi_master['gdhi_per_head'] = gdhi_master['gdhi_per_head'].round(0)
gdhi_master['gdhi_index'] = gdhi_master['gdhi_index'].round(1)
gdhi_master['gdhi_growth'] = gdhi_master['gdhi_growth'].round(1)

# 8. 对齐 Charging_Port 的地理顺序
charging = pd.read_csv('data/data_raw/Charging_Port.csv')
charging.columns = [c.strip() for c in charging.columns]
ons_order = charging['ONS_code'].str.strip().unique().tolist()
order_map = {code: i for i, code in enumerate(ons_order)}

gdhi_master['sort_rank'] = gdhi_master['ONS_code'].str.strip().map(order_map)
gdhi_final = gdhi_master.sort_values(by=['sort_rank', 'year']).drop(columns=['sort_rank'])

# 9. 导出结果
gdhi_final.to_csv('data/data_cleaned/GDHI_Final_Data_cleaned.csv', index=False)

print(gdhi_final.head(15)) # 预览前 15 行，应该能看到 UK 和 GB 两个汇总行

       ONS_code         ONS_geo Country Region Geo_level  year  gdhi_per_head  \
4693  K02000001  United Kingdom     NaN    NaN  National  2011        16413.0   
4694  K02000001  United Kingdom     NaN    NaN  National  2012        17034.0   
4695  K02000001  United Kingdom     NaN    NaN  National  2013        17677.0   
4696  K02000001  United Kingdom     NaN    NaN  National  2014        18248.0   
4697  K02000001  United Kingdom     NaN    NaN  National  2015        19244.0   
4698  K02000001  United Kingdom     NaN    NaN  National  2016        19474.0   
4699  K02000001  United Kingdom     NaN    NaN  National  2017        19936.0   
4700  K02000001  United Kingdom     NaN    NaN  National  2018        20704.0   
4701  K02000001  United Kingdom     NaN    NaN  National  2019        21302.0   
4702  K02000001  United Kingdom     NaN    NaN  National  2020        21255.0   
4703  K02000001  United Kingdom     NaN    NaN  National  2021        22011.0   
4704  K02000001  United King

In [17]:
# 1. 读取充电桩原始数据
charging_raw = pd.read_csv('data/data_raw/Charging_Port.csv')

# 2. 清洗列名
charging_raw.columns = [c.strip() for c in charging_raw.columns]

# 3. 识别代表年度汇总的列 (取每年的最后一个季度，即 Oct)
oct_cols = [c for c in charging_raw.columns if 'Oct' in c]

# 4. 提取地理信息列
geo_cols = ['ONS_code', 'ONS_geo', 'Country', 'Region', 'Geo_level']

# 5. 逆透视
charging_long = pd.melt(
    charging_raw,
    id_vars=geo_cols,
    value_vars=oct_cols,
    var_name='raw_date',
    value_name='charging_ports'
)

# 6. 从日期字符串提取年份
charging_long['year'] = charging_long['raw_date'].str.extract(r'(\d{2})').astype(int) + 2000

# 7. 数据清洗：处理千分位逗号、空值以及特殊标记 '[x]'
def clean_numeric(val):
    if pd.isna(val): return 0.0
    val_str = str(val).strip().replace(',', '')
    # [x] 是 ONS 常用的受限或不适用数据标记，将其设为 0
    if val_str in ['[x]', 'nan', '', '-']:
        return 0.0
    try:
        return float(val_str)
    except:
        return 0.0

charging_long['charging_ports'] = charging_long['charging_ports'].apply(clean_numeric)

# 8. 过滤无效行：保留合法的 LAD 级及以上层级
charging_final = charging_long[charging_long['ONS_code'].astype(str).str.len() >= 9].copy()

# 9. 最终格式整理
charging_final = charging_final[geo_cols + ['year', 'charging_ports']]
charging_final = charging_final.sort_values(['ONS_code', 'year'])

# 导出清洗后的中间表
charging_final.to_csv('data/data_cleaned/Charging_Port_Yearly_Cleaned.csv', index=False)

In [18]:
# 1. 加载 VEH0105 原始数据
veh_raw = pd.read_csv('data/data_raw/veh0105.csv', skiprows=4)

# 2. 清洗列名
veh_raw.columns = [c.strip() for c in veh_raw.columns]

# 3. 筛选所有能源类型的乘用车总数
# BodyType: Cars (乘用车)
# Fuel: Total (所有能源总计)
# Keepership: Total (私家车与公司车总计)
veh_filtered = veh_raw[
    (veh_raw['BodyType'] == 'Cars') &
    (veh_raw['Fuel'] == 'Total') &
    (veh_raw['Keepership'] == 'Total')
].copy()

# 4. 动态识别年度代表列
all_cols = veh_raw.columns.tolist()
# 选取所有年份的 Q4，并单独加入 2025 Q3
target_cols = [c for c in all_cols if 'Q4' in c]
if '2025 Q3' in all_cols:
    target_cols.append('2025 Q3')

# 5. 逆透视
veh_long = pd.melt(
    veh_filtered,
    id_vars=['ONS Code'],
    value_vars=target_cols,
    var_name='raw_quarter',
    value_name='veh_total'
)

# 6. 从季度字符串中提取年份数字
veh_long['year'] = veh_long['raw_quarter'].str.extract(r'(\d{4})').astype(int)

# 7. 数据清洗与单位还原
def clean_vehicle_count(val):
    val_str = str(val).strip().lower().replace(',', '')
    # 处理 ONS 特殊标记：[x] 不可用, [z] 不适用, [low] 极低值
    if any(mark in val_str for mark in ['[x]', '[z]', '[low]', 'no data', 'nan']):
        return 0.0
    try:
        return float(val_str)
    except:
        return 0.0

# 乘以 1000 还原为真实车辆数
veh_long['veh_total'] = veh_long['veh_total'].apply(clean_vehicle_count) * 1000
veh_long['veh_total'] = veh_long['veh_total'].round(0).astype(int)

# 8. 规范化列名，准备合并
# 去除空格
veh_long['ONS_code'] = veh_long['ONS Code'].astype(str).str.strip()

# 排除 [z]、[x] 以及其他非标准编码
veh_final = veh_long[
    (veh_long['ONS_code'].str.len() >= 9) &
    (~veh_long['ONS_code'].isin(['[z]', '[x]', 'nan', 'Unknown']))
].copy()

# 9. 最终筛选列
veh_final = veh_final[['ONS_code', 'year', 'veh_total']]

# 排序并检查
veh_final = veh_final.sort_values(['ONS_code', 'year'])
print(veh_final.head())
veh_final.to_csv('data/data_cleaned/VEH0105_Total_Cars_Cleaned.csv', index=False)

       ONS_code  year  veh_total
6711  E06000001  2009      35100
6264  E06000001  2010      34900
5817  E06000001  2011      35100
5370  E06000001  2012      35500
4923  E06000001  2013      36100


In [19]:
# 1. 加载 VEH0132 原始数据 (跳过前 4 行说明)
veh0132_raw = pd.read_csv('data/data_raw/veh0132.csv', skiprows=4)
veh0132_raw.columns = [c.strip() for c in veh0132_raw.columns]

# 2. 核心筛选：
# Keepership: Total (确保包含私人和公司车辆)
# Fuel: 提取 'BATTERY ELECTRIC' (ZEV核心) 和 'Total' (所有超低排放车 ULEV)
veh_filtered = veh0132_raw[
    (veh0132_raw['Keepership'] == 'Total') &
    (veh0132_raw['Fuel'].isin(['BATTERY ELECTRIC', 'Total']))
].copy()

# 3. 动态识别年度代表列 (2025取Q3, 其余年份取Q4)
all_cols = veh0132_raw.columns.tolist()
target_cols = [c for c in all_cols if 'Q4' in c]
if '2025 Q3' in all_cols:
    target_cols.append('2025 Q3')

# 4. 逆透视
veh_long = pd.melt(
    veh_filtered,
    id_vars=['ONS Code', 'Fuel'],
    value_vars=target_cols,
    var_name='raw_quarter',
    value_name='count_raw'
)

# 5. 提取年份
veh_long['year'] = veh_long['raw_quarter'].str.extract(r'(\d{4})').astype(int)

# 6. 数值清洗
def clean_numeric(val):
    val_str = str(val).strip().lower().replace(',', '')
    if any(mark in val_str for mark in ['[x]', '[z]', '[low]', 'no data', 'nan']):
        return 0.0
    try:
        return float(val_str)
    except:
        return 0.0

# 转换为整数
veh_long['veh_count'] = veh_long['count_raw'].apply(clean_numeric).round(0).astype(int)

# 7. 转换为指标宽表
veh_pivot = veh_long.pivot_table(
    index=['ONS Code', 'year'],
    columns='Fuel',
    values='veh_count',
    aggfunc='sum'
).reset_index()

# 8. 规范化列名
veh_pivot = veh_pivot.rename(columns={
    'BATTERY ELECTRIC': 'veh_bev', # 纯电动(ZEV核心)
    'Total': 'veh_ulev_total',    # 超低排放总计(含PHEV)
    'ONS Code': 'ONS_code'
})

# 9. 过滤 ONS_code (剔除 [z] 和无效编码)
veh_pivot['ONS_code'] = veh_pivot['ONS_code'].astype(str).str.strip()
veh_final = veh_pivot[
    (veh_pivot['ONS_code'].str.len() >= 9) &
    (~veh_pivot['ONS_code'].isin(['[z]', '[x]', 'nan']))
].copy()

# 10. 排序并导出
veh_final = veh_final[['ONS_code', 'year', 'veh_bev', 'veh_ulev_total']]
veh_final = veh_final.sort_values(['ONS_code', 'year'])
veh_final.to_csv('data/data_cleaned/VEH0132_EV_Stock_cleaned.csv', index=False)

print(veh_final.head())

Fuel   ONS_code  year  veh_bev  veh_ulev_total
0     E06000001  2011        5               7
1     E06000001  2012        7               9
2     E06000001  2013        6               8
3     E06000001  2014       10              16
4     E06000001  2015       15              23


In [20]:
# 1. 加载全部清洗过的数据集
geo_lookup = pd.read_csv('data/data_cleaned/geo_lookup.csv')
df_gdhi = pd.read_csv('data/data_cleaned/GDHI_Final_Data_cleaned.csv')
df_pop = pd.read_csv('data/data_cleaned/population_cleaned.csv').rename(columns={'ons_code': 'ONS_code'})
df_veh_total = pd.read_csv('data/data_cleaned/VEH0105_Total_Cars_Cleaned.csv')
df_veh_ev = pd.read_csv('data/data_cleaned/VEH0132_EV_Stock_cleaned.csv')
df_charging = pd.read_csv('data/data_cleaned/Charging_Port_Yearly_Cleaned.csv')

# 2. 构建主面板(ONS编码 x 2011-2025年份)
# 这确保了即便某些年份数据缺失，地理结构依然完整
years = pd.DataFrame({'year': range(2011, 2026)})
skeleton = geo_lookup.assign(key=1).merge(years.assign(key=1), on='key').drop('key', axis=1)

# 3. 顺序合并
master = skeleton.copy()
master = pd.merge(master, df_gdhi[['ONS_code', 'year', 'gdhi_per_head', 'gdhi_index', 'gdhi_growth']], on=['ONS_code', 'year'], how='left')
master = pd.merge(master, df_pop[['ONS_code', 'year', 'pop_value']], on=['ONS_code', 'year'], how='left')
master = pd.merge(master, df_veh_total[['ONS_code', 'year', 'veh_total']], on=['ONS_code', 'year'], how='left')
master = pd.merge(master, df_veh_ev[['ONS_code', 'year', 'veh_bev', 'veh_ulev_total']], on=['ONS_code', 'year'], how='left')
master = pd.merge(master, df_charging[['ONS_code', 'year', 'charging_ports']], on=['ONS_code', 'year'], how='left')

# 4. 人口数据平滑处理
# 使用 2024 年的人口数据向前填充给 2025 年，以保住 2025 年的人均计算指标
master = master.sort_values(['ONS_code', 'year'])
master['pop_value'] = master.groupby('ONS_code')['pop_value'].ffill()

# 5. 计算衍生核心研究指标
# EV 渗透率 (%)
master['ev_penetration'] = (master['veh_ulev_total'] / master['veh_total'] * 100).round(2)
# 每千人拥有的充电桩数
master['ports_per_1k_pop'] = (master['charging_ports'] / (master['pop_value'] / 1000)).round(2)
# 车桩比
master['bev_to_port_ratio'] = (master['veh_ulev_total'] / master['charging_ports']).round(2)

# 6. 最终格式整理与地理排序
# 按照 geo_lookup 的原始层级顺序进行排序
order_rank = {code: i for i, code in enumerate(geo_lookup['ONS_code'].unique())}
master['geo_rank'] = master['ONS_code'].map(order_rank)
final_cols = ['ONS_code', 'ONS_geo', 'Geo_level', 'Country', 'Region', 'year',
              'gdhi_per_head', 'gdhi_index', 'gdhi_growth',
              'pop_value', 'veh_total', 'veh_bev', 'veh_ulev_total', 'charging_ports',
              'ev_penetration', 'ports_per_1k_pop', 'bev_to_port_ratio']
master_final = master.sort_values(['geo_rank', 'year'])[final_cols]

master_final_cleaned = master_final[
    (~master_final['ONS_geo'].str.contains('abolished', case=False, na=False)) &
    (master_final['ONS_code'].notna())
].copy()

# 7. 导出最终成果
master_final_cleaned.to_csv('data/data_cleaned/Master_Panel_Data_2011_2025.csv', index=False)

In [21]:
# 1. 处理北爱尔兰人口数据
ni_raw = pd.read_csv('data/data_raw/north_Ireland_population.csv')

# 提取起初人口 (Mid-year population)
ni_pop = ni_raw[ni_raw['category'] == 'Starting population'].copy()

# 年份映射字典 (如 "2011/2012" 映射为 2011)
ni_year_map = {f"{y}/{y+1}": y for y in range(2011, 2024)}

# 提取所需的列
ni_cols_to_keep = ['area_code'] + list(ni_year_map.keys())
ni_pop = ni_pop[ni_cols_to_keep]

# 逆透视
ni_long = pd.melt(ni_pop, id_vars=['area_code'], var_name='raw_year', value_name='pop_value')
ni_long['year'] = ni_long['raw_year'].map(ni_year_map)
ni_long['ONS_code'] = ni_long['area_code'].str.strip()

# 清洗数值：去除逗号并转换为浮点数
ni_long['new_pop_value'] = ni_long['pop_value'].astype(str).str.replace(',', '').astype(float)
ni_final = ni_long[['ONS_code', 'year', 'new_pop_value']].dropna()


# 2. 处理苏格兰人口数据 (2011-2022)
# 设置文件路径
file_path = 'data/data_raw/mid-year-pop-est-revised-2011-2022-data-tab1.xlsx'

# 一次性读取所有 Sheet
all_sheets_dict = pd.read_excel(file_path, sheet_name=None, skiprows=3)

# 准备一个空列表来存放处理后的各年份数据
scot_data_list = []

# 遍历字典提取有用年份的数据
for sheet_name, df_scot in all_sheets_dict.items():
    # 只处理名称为 2011 到 2022 的 sheet (过滤掉 Cover sheet, Notes 等)
    if sheet_name.isdigit() and 2011 <= int(sheet_name) <= 2022:
        year = int(sheet_name)

        # 模糊匹配 Area code 列名（处理换行符）
        code_col = [c for c in df_scot.columns if 'Area code' in c][0]

        # 筛选区域为 S12 开头 (LAD级) 且 Sex 为 'Persons'
        df_valid = df_scot[
            (df_scot[code_col].astype(str).str.startswith(('S12', 'S92'))) &
            (df_scot['Sex'] == 'Persons')
        ].copy()

        df_valid['ONS_code'] = df_valid[code_col].str.strip()
        df_valid['new_pop_value'] = df_valid['All ages'].astype(str).str.replace(',', '').astype(float)
        df_valid['year'] = year

        scot_data_list.append(df_valid[['ONS_code', 'year', 'new_pop_value']])

# 合并为一个完整的大表
scot_final = pd.concat(scot_data_list, ignore_index=True)


extra_pop = pd.concat([ni_final, scot_final], ignore_index=True)

# 3. 连接到总面板数据
master = pd.read_csv('data/data_cleaned/Master_Panel_Data_2011_2025.csv')

# 恢复绝对顺序
master['original_row_id'] = master.index

# 将补丁包通过 Left Join 挂载到主表上
master = pd.merge(master, extra_pop, on=['ONS_code', 'year'], how='left')

# 更新 pop_value：如果原 pop_value 为 NaN，则用提取的 new_pop_value 填补
master['pop_value'] = master['pop_value'].fillna(master['new_pop_value'])

# 清理不再需要的中间列
master = master.drop(columns=['new_pop_value'])

# 补齐时间序列末尾的缺失值
# 针对苏格兰 (缺2023-2025) 和 北爱尔兰 (缺2024-2025)
master = master.sort_values(['ONS_code', 'year'])
master['pop_value'] = master.groupby('ONS_code')['pop_value'].ffill()

# 按照合并前的初始行号重新排序
master_final = master.sort_values('original_row_id').drop(columns=['original_row_id'])


# 4. 导出
output_file = 'data/data_cleaned/Master_Panel_Data.csv'
master_final.to_csv(output_file, index=False)

print(master_final[master_final['ONS_code'].str.startswith(('N', 'S'), na=False)][['ONS_code', 'ONS_geo', 'year', 'pop_value']].head(5))

       ONS_code   ONS_geo  year  pop_value
5400  S92000003  Scotland  2011  5299900.0
5401  S92000003  Scotland  2012  5308200.0
5402  S92000003  Scotland  2013  5316800.0
5403  S92000003  Scotland  2014  5331400.0
5404  S92000003  Scotland  2015  5350600.0


In [22]:
# 1. 加载数据
master = pd.read_csv('data/data_cleaned/Master_Panel_Data.csv')
cp_10k = pd.read_csv('data/data_raw/Charging_Port_Per_10k_Population.csv')

# 2. 清洗列名
cp_10k.columns = [c.strip() for c in cp_10k.columns]

# 3. 识别代表年度汇总的列 (寻找包含 'Oct' 的所有列)
oct_cols = [c for c in cp_10k.columns if 'Oct' in c]

# 4. 逆透视
# 提取地理信息列
geo_cols = ['ONS_code', 'ONS_geo', 'Geo_level'] # 根据 10k 表的实际列名
cp_10k_long = pd.melt(
    cp_10k,
    id_vars=['ONS_code'], # 只要 ONS_code 用于后续匹配
    value_vars=oct_cols,
    var_name='raw_date',
    value_name='ports_per_10k_pop'
)

# 5. 从日期字符串动态提取年份
# 提取列名中的两位数字
cp_10k_long['year'] = cp_10k_long['raw_date'].str.extract(r'(\d{2})').astype(int) + 2000

# 6. 准备合并
cp_10k_long['ONS_code'] = cp_10k_long['ONS_code'].astype(str).str.strip()
cp_10k_final = cp_10k_long[['ONS_code', 'year', 'ports_per_10k_pop']]

# 7. 缝合至主表
master['original_order'] = range(len(master))

# 删除旧的 1k 密度列
if 'ports_per_1k_pop' in master.columns:
    master = master.drop(columns=['ports_per_1k_pop'])

# 左连接
master_updated = pd.merge(master, cp_10k_final, on=['ONS_code', 'year'], how='left')

# 恢复排序
master_final = master_updated.sort_values('original_order').drop(columns=['original_order'])

# 调整列顺序
cols = master_final.columns.tolist()
if 'ports_per_10k_pop' in cols:
    cols.remove('ports_per_10k_pop')
    if 'ev_penetration' in cols:
        idx = cols.index('ev_penetration') + 1
        cols.insert(idx, 'ports_per_10k_pop')
        master_final = master_final[cols]

if 'bev_to_port_ratio' in master_final.columns:
        master_final = master_final.rename(columns={'bev_to_port_ratio': 'ulev_to_port_ratio'})

# 8. 导出
master_final.to_csv('data/data_cleaned/Master_Panel_Data_V2_Cleaned.csv', index=False)
print(master_final[master_final['year'] == 2024][['ONS_geo', 'year', 'ports_per_10k_pop']].head())

           ONS_geo  year ports_per_10k_pop
13  United Kingdom  2024             103.6
28   Great Britain  2024             105.7
43         England  2024             105.7
58      North East  2024              85.5
73   County Durham  2024                68


In [23]:
# 1. 加载主表和分级字典
master = pd.read_csv('data/data_cleaned/Master_Panel_Data_V2_Cleaned.csv')
lookup = pd.read_csv('data/data_cleaned/geo_lookup.csv')

# 记录原始顺序
master['original_index'] = master.index

# 2. 提取最基础的 LAD 级人口数据（作为聚合的原子数据）
df_lad = master[master['Geo_level'] == 'LAD'][['ONS_code', 'year', 'pop_value']].copy()

# 3. 关联地理字典，获取每个 LAD 的所有父级行政名称
df_lad_hierarchy = df_lad.merge(
    lookup[['ONS_code', 'p_uk', 'p_gb', 'Country', 'Region', 'p_cty']],
    on='ONS_code',
    how='left'
)

# 定义聚合函数：按父级名称和年份计算人口总和
def get_agg(df, group_col, level_name):
    agg = df.groupby([group_col, 'year'])['pop_value'].sum().reset_index()
    agg.columns = ['ONS_geo', 'year', 'calculated_pop']
    agg['Geo_level'] = level_name
    return agg

# 分层级计算
agg_uk      = get_agg(df_lad_hierarchy, 'p_uk', 'National')
agg_gb      = get_agg(df_lad_hierarchy, 'p_gb', 'National')
agg_country = get_agg(df_lad_hierarchy, 'Country', 'Country')
agg_region  = get_agg(df_lad_hierarchy, 'Region', 'Region')
agg_county  = get_agg(df_lad_hierarchy, 'p_cty', 'County')

# 合并所有计算出的汇总数据
all_aggs = pd.concat([agg_uk, agg_gb, agg_country, agg_region, agg_county], ignore_index=True)
all_aggs = all_aggs.dropna(subset=['ONS_geo']) # 剔除无效行

# 4. 将汇总结果缝合回主表
master = master.merge(
    all_aggs,
    on=['ONS_geo', 'Geo_level', 'year'],
    how='left'
)

# 填补空缺值：如果原 pop_value 为空，则填入计算出的汇总值
master['pop_value'] = master['pop_value'].fillna(master['calculated_pop'])

# 5. 清理并恢复顺序
master_final = master.sort_values('original_index').drop(columns=['original_index', 'calculated_pop'])

# 导出
master_final.to_csv('data/data_cleaned/Master_Panel_Data_V3.csv', index=False)

In [24]:
# 1. 加载主数据表 V3
master = pd.read_csv('data/data_cleaned/Master_Panel_Data_V3.csv')

# 定义需要补齐的构成国
target_countries = ['Scotland', 'Northern Ireland']

# 2. 提取基础 LAD 数据进行加权计算
# 仅选择 2011-2023 年间（官方 GDHI 统计期）且人口与 GDHI 均不为空的行
lad_data = master[
    (master['Geo_level'] == 'LAD') &
    (master['Country'].isin(target_countries)) &
    (master['gdhi_per_head'].notna()) &
    (master['pop_value'].notna())
].copy()

# 计算人口加权分子：总收入 = 人均收入 * 人口
lad_data['w_head'] = lad_data['gdhi_per_head'] * lad_data['pop_value']
lad_data['w_index'] = lad_data['gdhi_index'] * lad_data['pop_value']

# 3. 按国家和年份执行聚合
agg_country = lad_data.groupby(['Country', 'year']).agg({
    'w_head': 'sum',
    'w_index': 'sum',
    'pop_value': 'sum'
}).reset_index()

# 计算加权后的 Country 级别人均值和指数
agg_country['new_head'] = agg_country['w_head'] / agg_country['pop_value']
agg_country['new_index'] = agg_country['w_index'] / agg_country['pop_value']

# 4. 回填空缺值
# 仅填补主表中 Geo_level 为 Country 且数据为空的苏格兰和北爱尔兰行
for country in target_countries:
    for year in agg_country['year'].unique():
        # 获取计算出的补丁数据
        patch = agg_country[(agg_country['Country'] == country) & (agg_country['year'] == year)]

        if not patch.empty:
            # 定位主表中的对应单元格
            mask = (master['Geo_level'] == 'Country') & \
                   (master['ONS_geo'] == country) & \
                   (master['year'] == year)

            # 填补人均 GDHI 和 GDHI 指数
            master.loc[mask, 'gdhi_per_head'] = patch['new_head'].round(0).values[0]
            master.loc[mask, 'gdhi_index'] = patch['new_index'].round(1).values[0]

# 5. 重新计算受影响行的年度增长率 (gdhi_growth)
# 针对补齐后的构成国，重新生成基于人均值的百分比变化
for country in target_countries:
    mask = (master['Geo_level'] == 'Country') & (master['ONS_geo'] == country)
    # 确保按年份排序后计算
    country_series = master[mask].sort_values('year')
    master.loc[mask, 'gdhi_growth'] = (country_series['gdhi_per_head'].pct_change(fill_method=None) * 100).round(1)

master.to_csv('data/data_cleaned/Master_Panel_Data_Final.csv', index=False)